# Load One Raw 10x Sample to AnnData

This notebook starts with one per-sample 10x matrix for a clean Scanpy import check.

After this works, you can loop over all samples.

In [ ]:
from pathlib import Path
import pandas as pd
import scanpy as sc

In [ ]:
# DIV30-only: pick one run sample from 9853-MW-1 ... 9853-MW-6.
SAMPLE_DIV = "DIV30"
SAMPLE_RUN_ID = "9853-MW-1"

DIV30_PER_SAMPLE_ROOT = Path(
    "/nfs/turbo/umms-parent/Manny_test/9583-MW-reanalysis/outs/per_sample_outs"
 )

SAMPLE_MATRIX_DIR = (
    DIV30_PER_SAMPLE_ROOT
    / SAMPLE_RUN_ID
    / "count"
    / "sample_filtered_feature_bc_matrix"
 )

print("Loading from:", SAMPLE_MATRIX_DIR)
sample_adata = sc.read_10x_mtx(SAMPLE_MATRIX_DIR, var_names="gene_symbols", make_unique=True)

Loading from: /nfs/turbo/umms-parent/Manny_test/9583-MW-reanalysis/outs/per_sample_outs/9853-MW-1/count/sample_filtered_feature_bc_matrix


In [ ]:
# Requested DIV30-only metadata map (first six samples only).
SAMPLE_TO_BIOLABEL = {
    "DIV30": {
        "9853-MW-1": "H9_rep1",
        "9853-MW-2": "H9_rep2",
        "9853-MW-3": "79B_rep1",
        "9853-MW-4": "79B_rep2",
        "9853-MW-5": "2E_rep1",
        "9853-MW-6": "2E_rep2",
    }
}

if SAMPLE_RUN_ID not in SAMPLE_TO_BIOLABEL["DIV30"]:
    raise ValueError(
        f"SAMPLE_RUN_ID={SAMPLE_RUN_ID} is not in the allowed DIV30 set: "
        f"{sorted(SAMPLE_TO_BIOLABEL['DIV30'].keys())}"
    )

sample_adata.obs["DIV"] = SAMPLE_DIV
sample_adata.obs["run_sample_id"] = SAMPLE_RUN_ID
sample_adata.obs["biological_label"] = SAMPLE_TO_BIOLABEL["DIV30"][SAMPLE_RUN_ID]

print("Loaded sample shape:", sample_adata.shape)
print("DIV:", SAMPLE_DIV)
print("run_sample_id:", SAMPLE_RUN_ID)
print("biological_label:", sample_adata.obs["biological_label"].iloc[0])
print("obs columns:", list(sample_adata.obs.columns))

Loaded sample shape: (18047, 18082)
DIV: DIV30
run_sample_id: 9853-MW-1
biological_label: H9_rep1
obs columns: ['DIV', 'run_sample_id', 'biological_label']


In [ ]:
# Quick object preview
sample_adata

AnnData object with n_obs × n_vars = 18047 × 18082
    obs: 'DIV', 'run_sample_id', 'biological_label'
    var: 'gene_ids', 'feature_types'

In [ ]:
# Compute Scanpy QC metrics for this sample.
sample_adata.var["mt"] = sample_adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    sample_adata,
    qc_vars=["mt"],
    percent_top=[20],
    log1p=True,
    inplace=True,
 )

qc_report = pd.Series(
    {
        "n_cells": int(sample_adata.n_obs),
        "n_genes": int(sample_adata.n_vars),
        "median_total_counts": float(sample_adata.obs["total_counts"].median()),
        "median_n_genes_by_counts": float(sample_adata.obs["n_genes_by_counts"].median()),
        "median_pct_counts_mt": float(sample_adata.obs["pct_counts_mt"].median()),
    },
    name="value",
 )
display(qc_report.to_frame())

                              DIV run_sample_id biological_label
AAACAAGCAAGATAAGACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
AAACAAGCAAGCCTAAACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
AAACAAGCAAGGCCATACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
AAACAAGCAATATGGTACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
AAACAAGCACTAACGAACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
...                           ...           ...              ...
TTTGTGAGTATGTTTGACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
TTTGTGAGTCCGCTAAACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
TTTGTGAGTCCTGAGCACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
TTTGTGAGTTGCTGTGACTTTAGG-1  DIV30     9853-MW-1          H9_rep1
TTTGTGAGTTGTCCAGACTTTAGG-1  DIV30     9853-MW-1          H9_rep1

[18047 rows x 3 columns]


In [ ]:
# Minimal metadata previews used in QC reporting
display(sample_adata.obs[["DIV", "run_sample_id", "biological_label"]].head())
display(sample_adata.var.head())

,gene_ids,feature_types
SAMD11,SAMD11,Gene Expression
NOC2L,NOC2L,Gene Expression
KLHL17,KLHL17,Gene Expression
PLEKHN1,PLEKHN1,Gene Expression
PERM1,PERM1,Gene Expression
...,...,...
MT-ND4L,MT-ND4L,Gene Expression
MT-ND4,MT-ND4,Gene Expression
MT-ND5,MT-ND5,Gene Expression
MT-ND6,MT-ND6,Gene Expression


In [ ]:
# QC plots (Scanpy): distributions of counts, genes, and mitochondrial fraction.
sc.pl.violin(
    sample_adata,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
 )

<Compressed Sparse Column sparse matrix of dtype 'float32'
	with 68720321 stored elements and shape (18047, 18082)>

In [ ]:
sc.pl.scatter(sample_adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")

,SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,AGRN,RNF223,C1orf159,...,MT-ND2,MT-CO2,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB
AAACAAGCAAGATAAGACTTTAGG-1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0
AAACAAGCAAGCCTAAACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,1.0,2.0,1.0,1.0,3.0,0.0,1.0,0.0
AAACAAGCAAGGCCATACTTTAGG-1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,5.0,10.0,4.0,8.0,1.0,3.0,7.0,4.0,0.0,2.0
AAACAAGCAATATGGTACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,...,8.0,12.0,14.0,13.0,7.0,5.0,15.0,1.0,1.0,4.0
AAACAAGCACTAACGAACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,2.0,5.0,6.0,11.0,1.0,4.0,8.0,2.0,0.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGAGTATGTTTGACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,9.0,9.0,8.0,4.0,2.0,6.0,3.0,2.0,3.0
TTTGTGAGTCCGCTAAACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,10.0,26.0,23.0,26.0,5.0,9.0,20.0,6.0,4.0,13.0
TTTGTGAGTCCTGAGCACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
TTTGTGAGTTGCTGTGACTTTAGG-1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,1.0,...,0.0,2.0,2.0,0.0,0.0,2.0,3.0,1.0,0.0,4.0


In [ ]:
# Optional: export a one-sample QC report table for records.
qc_out = Path("qc_report_div30_one_sample.tsv")
sample_adata.obs[[
    "DIV",
    "run_sample_id",
    "biological_label",
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
]].to_csv(qc_out, sep="\t", index=True)
print("Wrote", qc_out.resolve())